# 🎭 Six-Way Emotion Recognition: From Recurrent Nets to Transformers

**Author:** ML Engineering Notebook — Applied NLP Track
**Dataset:** GoEmotions (Demszky et al., Google Research)
**Target labels:** `joy`, `sadness`, `anger`, `fear`, `surprise`, `disgust`

---

### 🎯 What this notebook does

This project walks through a full applied-NLP workflow for **single-label, six-class emotion
recognition**, built to let us compare *how much representational power actually matters* —
starting from a plain recurrent baseline and ending with a fine-tuned Transformer.

We deliberately keep four model families in the ring at once:

| Family | Model | Why it's here |
|---|---|---|
| Recurrent | `LSTM` | Classic sequence baseline |
| Recurrent | `GRU` | Lighter-weight recurrent alternative |
| Recurrent + Attention | `BiLSTM` + hand-rolled additive attention | Interpretability + context in both directions |
| Transformer | `DistilBERT` (fine-tuned) | Modern pretrained baseline |

### 🧭 Pipeline at a glance

```mermaid
flowchart LR
    A[Raw GoEmotions] --> B[Label Collapse: 27 -> 6]
    B --> C[Text Normalization]
    C --> D[Vocabulary + Padding]
    D --> E1[LSTM]
    D --> E2[GRU]
    D --> E3[BiLSTM + Attention]
    C --> F[DistilBERT Tokenizer]
    F --> G[DistilBERT Fine-tune]
    E1 --> H[Benchmark + Confusion Matrices]
    E2 --> H
    E3 --> H
    G --> H
    H --> I[Error Forensics]
    H --> J[Interactive Inference Playground]
```

### 📦 What you'll get out of running this top-to-bottom

- A cleaned, six-class version of GoEmotions with full EDA
- Four trained classifiers with saved checkpoints
- A from-scratch additive attention layer with token-level heatmaps
- A single benchmark table (accuracy, macro-P/R/F1, latency, parameter count, memory)
- Confusion matrices + per-class reports for every model
- A small playground cell where you type a sentence and see all four models vote


## 1 · Environment Setup

We install only what isn't already available in a fresh Colab / Jupyter kernel. Splitting the
install from the imports keeps the "did my kernel restart" debugging loop short.

In [ ]:
# Runtime dependencies (safe to re-run; pip no-ops if already satisfied)
%pip install -q datasets evaluate transformers accelerate wordcloud contractions


### 1.1 Core imports

Grouped by purpose (stdlib → data → viz → deep learning → transformers) rather than
alphabetically — it's easier to scan when you're hunting for "wait, where did I import `re`?".

In [ ]:
# --- stdlib ---
import os
import re
import json
import time
import random
import string
import warnings
from collections import Counter

warnings.filterwarnings("ignore")


In [ ]:
# --- data wrangling ---
import numpy as np
import pandas as pd
from datasets import load_dataset


In [ ]:
# --- visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110


In [ ]:
# --- classical deep learning (recurrent models) ---
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, mixed_precision
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
# --- transformer fine-tuning ---
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate


In [ ]:
# --- misc quality-of-life ---
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)


## 2 · Project Configuration

One cell, one source of truth. Every hyperparameter used later reads from `CFG` so re-running
an experiment with different settings never means hunting through 40 cells.

In [ ]:
class CFG:
    # reproducibility
    SEED = 42

    # data
    HF_DATASET_NAME = "google-research-datasets/go_emotions"
    HF_DATASET_CONFIG = "simplified"

    # sequence processing
    MAX_VOCAB_SIZE = 20_000
    MAX_SEQ_LEN = 48
    OOV_TOKEN = "<OOV>"

    # embeddings — flip this one string to switch the whole pipeline
    EMBEDDING_SOURCE = "glove"          # "glove" or "fasttext"
    EMBEDDING_DIM = 100
    GLOVE_PATH_CANDIDATES = [
        "/kaggle/input/glove6b100dtxt/glove.6B.100d.txt",
        "/content/glove.6B.100d.txt",
        "glove.6B.100d.txt",
    ]
    FASTTEXT_PATH_CANDIDATES = [
        "/kaggle/input/fasttext-wikinews/wiki-news-300d-1M.vec",
        "/content/wiki-news-300d-1M.vec",
        "wiki-news-300d-1M.vec",
    ]

    # recurrent model hyperparameters
    RNN_UNITS = 128
    ATTENTION_UNITS = 64
    DROPOUT_RATE = 0.3
    BATCH_SIZE = 64
    EPOCHS = 12
    LEARNING_RATE = 1e-3

    # transformer hyperparameters
    TRANSFORMER_CHECKPOINT = "distilbert-base-uncased"
    TRANSFORMER_MAX_LEN = 64
    TRANSFORMER_BATCH_SIZE = 32
    TRANSFORMER_EPOCHS = 3
    TRANSFORMER_LR = 2e-5

    # artifacts
    ARTIFACT_DIR = "artifacts"
    FIGURE_DIR = "figures"


os.makedirs(CFG.ARTIFACT_DIR, exist_ok=True)
os.makedirs(CFG.FIGURE_DIR, exist_ok=True)


In [ ]:
def fix_random_seeds(seed: int) -> None:
    '''Pin every RNG we touch so re-runs are comparable.'''
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


fix_random_seeds(CFG.SEED)


In [ ]:
# Mixed precision speeds up training substantially on modern GPUs; harmless on CPU.
gpu_devices = tf.config.list_physical_devices("GPU")
if gpu_devices:
    mixed_precision.set_global_policy("mixed_float16")
    print(f"GPU detected ({len(gpu_devices)}): mixed precision enabled.")
else:
    print("No GPU detected: training in float32 on CPU (slower, still correct).")


## 3 · Data Acquisition

GoEmotions ships pre-split into train/validation/test on the Hugging Face Hub, so we load it
directly rather than hand-rolling a split.

In [ ]:
raw_dataset = load_dataset(CFG.HF_DATASET_NAME, CFG.HF_DATASET_CONFIG)
raw_dataset


In [ ]:
df_train = raw_dataset["train"].to_pandas()
df_val   = raw_dataset["validation"].to_pandas()
df_test  = raw_dataset["test"].to_pandas()

print(f"train: {df_train.shape} | val: {df_val.shape} | test: {df_test.shape}")
df_train.head()


### 3.1 Structural audit

Before touching content, confirm the frame looks the way we expect: no surprise nulls,
sane text lengths, and label columns that match the docs.

In [ ]:
def audit_frame(df: pd.DataFrame, name: str) -> pd.Series:
    '''One-line health check: nulls, blanks, duplicate rows.'''
    report = {
        "rows": len(df),
        "null_text": df["text"].isna().sum(),
        "blank_text": (df["text"].str.strip() == "").sum(),
        "duplicate_rows": df.duplicated(subset=["text"]).sum(),
    }
    return pd.Series(report, name=name)


pd.concat([
    audit_frame(df_train, "train"),
    audit_frame(df_val, "validation"),
    audit_frame(df_test, "test"),
], axis=1)


In [ ]:
GOEMOTIONS_LABEL_NAMES = raw_dataset["train"].features["labels"].feature.names
print(f"{len(GOEMOTIONS_LABEL_NAMES)} fine-grained GoEmotions labels:")
print(GOEMOTIONS_LABEL_NAMES)


## 4 · Exploratory Data Analysis

We look at the data *before* collapsing labels, so we can see how skewed the raw 27-class
distribution already is — that context matters when we later decide how to handle imbalance
in the 6-class version.

In [ ]:
def explode_label_counts(df: pd.DataFrame, label_names: list[str]) -> pd.Series:
    '''GoEmotions is multi-label; explode to count raw label frequency.'''
    counter = Counter()
    for row_labels in df["labels"]:
        for idx in row_labels:
            counter[label_names[idx]] += 1
    return pd.Series(counter).sort_values(ascending=False)


raw_label_freq = explode_label_counts(df_train, GOEMOTIONS_LABEL_NAMES)
raw_label_freq


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=raw_label_freq.values, y=raw_label_freq.index, ax=ax, color="steelblue")
ax.set_title("Raw GoEmotions label frequency (train split, multi-label)")
ax.set_xlabel("Number of examples")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_DIR}/raw_label_frequency.png")
plt.show()


**Observation:** GoEmotions is heavily skewed toward `neutral` and `admiration`, while
emotions like `grief` or `pride` barely register. This is exactly why we collapse to six
broader, more balanced buckets in the next section rather than trying to model all 27 classes
directly.

In [ ]:
df_train["char_len"] = df_train["text"].str.len()
df_train["word_len"] = df_train["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df_train["char_len"], bins=50, ax=axes[0], color="teal")
axes[0].set_title("Character length distribution")
sns.histplot(df_train["word_len"], bins=30, ax=axes[1], color="coral")
axes[1].set_title("Word count distribution")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_DIR}/text_length_distributions.png")
plt.show()

print(df_train[["char_len", "word_len"]].describe())


**Observation:** most comments are short (Reddit-style), with word counts clustering
under ~20 tokens. This directly informs `CFG.MAX_SEQ_LEN = 48` — generous enough to avoid
truncating the long tail, without padding every batch to a huge, mostly-empty length.

In [ ]:
def build_wordcloud(texts: pd.Series, title: str) -> None:
    blob = " ".join(texts.sample(min(5000, len(texts)), random_state=CFG.SEED))
    cloud = WordCloud(width=1000, height=450, background_color="white",
                       colormap="viridis", max_words=150).generate(blob)
    plt.figure(figsize=(12, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.show()


build_wordcloud(df_train["text"], "Most frequent words — raw training text")


In [ ]:
def top_n_words(texts: pd.Series, n: int = 25) -> pd.Series:
    tokens = " ".join(texts).lower().split()
    tokens = [t.strip(string.punctuation) for t in tokens if t.strip(string.punctuation)]
    return pd.Series(Counter(tokens)).sort_values(ascending=False).head(n)


top_words = top_n_words(df_train["text"])
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(x=top_words.values, y=top_words.index, ax=ax, color="mediumpurple")
ax.set_title("Top 25 raw tokens (pre-cleaning, stopwords included)")
plt.tight_layout()
plt.show()


## 5 · Collapsing 27 Fine-Grained Labels into 6 Core Emotions

GoEmotions' 27 labels are useful for research but too fine-grained (and too imbalanced) for a
practical six-way classifier. We map each fine-grained label onto one of our six target
emotions, and **drop** anything that doesn't clearly belong (e.g. `neutral`, `admiration`,
`curiosity`, `realization`) rather than force-fitting it and adding noise.

The mapping choices below follow the emotional-family groupings used in Ekman's basic-emotion
taxonomy, which GoEmotions itself was partly designed around.

In [ ]:
TAXONOMY_MAP = {
    # joy family
    "joy": "joy", "amusement": "joy", "excitement": "joy", "gratitude": "joy",
    "love": "joy", "optimism": "joy", "relief": "joy", "pride": "joy",
    "admiration": "joy",
    # sadness family
    "sadness": "sadness", "grief": "sadness", "disappointment": "sadness",
    "remorse": "sadness",
    # anger family
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    # fear family
    "fear": "fear", "nervousness": "fear", "embarrassment": "fear",
    # surprise family
    "surprise": "surprise", "confusion": "surprise", "realization": "surprise",
    "curiosity": "surprise",
    # disgust family
    "disgust": "disgust",
    # everything else (neutral, approval, caring, desire) is intentionally dropped
}

TARGET_EMOTIONS = ["joy", "sadness", "anger", "fear", "surprise", "disgust"]
print(f"{len(TAXONOMY_MAP)}/{len(GOEMOTIONS_LABEL_NAMES)} fine-grained labels retained and mapped.")


In [ ]:
def collapse_to_six(df: pd.DataFrame, label_names: list[str]) -> pd.DataFrame:
    '''Keep only single-label rows whose one fine-grained label maps to a target emotion.

    Multi-label / ambiguous rows are dropped by design — training a *single-label* six-way
    classifier on ambiguous multi-emotion text would just inject label noise.
    '''
    def resolve(label_idxs):
        if len(label_idxs) != 1:
            return None
        fine_label = label_names[label_idxs[0]]
        return TAXONOMY_MAP.get(fine_label)

    out = df.copy()
    out["emotion"] = out["labels"].apply(resolve)
    out = out.dropna(subset=["emotion"]).reset_index(drop=True)
    return out[["text", "emotion"]]


clean_train = collapse_to_six(df_train, GOEMOTIONS_LABEL_NAMES)
clean_val   = collapse_to_six(df_val, GOEMOTIONS_LABEL_NAMES)
clean_test  = collapse_to_six(df_test, GOEMOTIONS_LABEL_NAMES)

print(f"Retained rows -> train: {len(clean_train)}, val: {len(clean_val)}, test: {len(clean_test)}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.countplot(data=clean_train, y="emotion", order=TARGET_EMOTIONS,
              palette="mako", ax=ax)
ax.set_title("Six-class distribution after taxonomy collapse (train)")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_DIR}/six_class_distribution.png")
plt.show()

clean_train["emotion"].value_counts().reindex(TARGET_EMOTIONS)


**Observation:** `joy` still dominates (it absorbed the largest number of fine-grained
labels), while `disgust` and `fear` remain comparatively rare. We keep this imbalance visible
rather than artificially rebalancing — and we lean on **macro-averaged** metrics later
specifically so rare classes aren't drowned out in the scoring.

## 6 · Text Normalization Pipeline

A single, composable `normalize_text` function — every step is its own small function so it's
easy to toggle stopword removal / lemmatization on or off without rewriting the pipeline.

In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
HTML_PATTERN = re.compile(r"<.*?>")
PUNCT_TABLE = str.maketrans("", "", string.punctuation)
MULTI_SPACE_PATTERN = re.compile(r"\s+")


def strip_urls(text: str) -> str:
    return URL_PATTERN.sub(" ", text)


def strip_html(text: str) -> str:
    return HTML_PATTERN.sub(" ", text)


def strip_punctuation(text: str) -> str:
    return text.translate(PUNCT_TABLE)


def strip_digits(text: str) -> str:
    return re.sub(r"\d+", " ", text)


def collapse_whitespace(text: str) -> str:
    return MULTI_SPACE_PATTERN.sub(" ", text).strip()


In [ ]:
import contractions as _contractions

def expand_contractions(text: str) -> str:
    return _contractions.fix(text)


def normalize_text(text: str, remove_stopwords: bool = False) -> str:
    '''Full cleaning pipeline, applied in a fixed, documented order.'''
    text = text.lower()
    text = expand_contractions(text)
    text = strip_urls(text)
    text = strip_html(text)
    text = strip_punctuation(text)
    text = strip_digits(text)
    text = collapse_whitespace(text)
    if remove_stopwords:
        text = " ".join(w for w in text.split() if w not in STOPWORDS)
    return text


In [ ]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "for", "and", "or", "but", "with", "at", "by",
}  # small, curated list — emotion words like "not" / "very" are deliberately kept

sample_before = clean_train["text"].sample(5, random_state=CFG.SEED)
for original in sample_before:
    print("BEFORE:", original)
    print("AFTER :", normalize_text(original))
    print("-" * 80)


In [ ]:
for split_df in (clean_train, clean_val, clean_test):
    split_df["clean_text"] = split_df["text"].apply(normalize_text)

clean_train = clean_train[clean_train["clean_text"].str.len() > 0].reset_index(drop=True)
clean_val   = clean_val[clean_val["clean_text"].str.len() > 0].reset_index(drop=True)
clean_test  = clean_test[clean_test["clean_text"].str.len() > 0].reset_index(drop=True)

clean_train[["text", "clean_text", "emotion"]].head()


## 7 · Tokenization, Vocabulary & Padding

The recurrent models (LSTM/GRU/BiLSTM) share one Keras `Tokenizer` and one padded-sequence
representation, fit **only on training text** to avoid any validation/test leakage into the
vocabulary.

In [ ]:
emotion_to_id = {name: i for i, name in enumerate(TARGET_EMOTIONS)}
id_to_emotion = {i: name for name, i in emotion_to_id.items()}

for split_df in (clean_train, clean_val, clean_test):
    split_df["label_id"] = split_df["emotion"].map(emotion_to_id)

emotion_to_id


In [ ]:
sequence_tokenizer = Tokenizer(num_words=CFG.MAX_VOCAB_SIZE, oov_token=CFG.OOV_TOKEN)
sequence_tokenizer.fit_on_texts(clean_train["clean_text"])

vocab_size = min(CFG.MAX_VOCAB_SIZE, len(sequence_tokenizer.word_index) + 1)
print(f"Fitted vocabulary: {len(sequence_tokenizer.word_index)} unique tokens "
      f"(capped to {vocab_size} for embedding matrix).")


In [ ]:
def texts_to_padded(texts: pd.Series) -> np.ndarray:
    sequences = sequence_tokenizer.texts_to_sequences(texts)
    return pad_sequences(sequences, maxlen=CFG.MAX_SEQ_LEN, padding="post", truncating="post")


X_train_seq = texts_to_padded(clean_train["clean_text"])
X_val_seq   = texts_to_padded(clean_val["clean_text"])
X_test_seq  = texts_to_padded(clean_test["clean_text"])

y_train = clean_train["label_id"].to_numpy()
y_val   = clean_val["label_id"].to_numpy()
y_test  = clean_test["label_id"].to_numpy()

X_train_seq.shape, X_val_seq.shape, X_test_seq.shape


## 8 · Pretrained Word Embeddings (GloVe / FastText)

Both loaders return the same shape of dictionary (`token -> vector`), so building the
embedding matrix is source-agnostic. Switch `CFG.EMBEDDING_SOURCE` between `"glove"` and
`"fasttext"` and re-run this section — nothing downstream needs to change.

In [ ]:
def load_glove_vectors(dim: int) -> dict[str, np.ndarray]:
    path = next((p for p in CFG.GLOVE_PATH_CANDIDATES if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError(
            "GloVe file not found. Download glove.6B.zip and point "
            "CFG.GLOVE_PATH_CANDIDATES at the extracted .100d.txt file."
        )
    vectors = {}
    with open(path, encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading GloVe"):
            parts = line.rstrip().split(" ")
            vectors[parts[0]] = np.asarray(parts[1:], dtype="float32")
    return vectors


def load_fasttext_vectors(dim: int) -> dict[str, np.ndarray]:
    path = next((p for p in CFG.FASTTEXT_PATH_CANDIDATES if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError(
            "FastText file not found. Download wiki-news-300d-1M.vec and point "
            "CFG.FASTTEXT_PATH_CANDIDATES at it."
        )
    vectors = {}
    with open(path, encoding="utf-8") as f:
        next(f)  # header line: n_words dim
        for line in tqdm(f, desc="Loading FastText"):
            parts = line.rstrip().split(" ")
            vectors[parts[0]] = np.asarray(parts[1:], dtype="float32")
    return vectors


EMBEDDING_LOADERS = {"glove": load_glove_vectors, "fasttext": load_fasttext_vectors}


In [ ]:
def build_embedding_matrix(word_index: dict, vocab_size: int, dim: int,
                            source: str) -> tuple[np.ndarray, float]:
    '''Returns (embedding_matrix, coverage_ratio).'''
    vectors = EMBEDDING_LOADERS[source](dim)
    matrix = np.random.normal(scale=0.1, size=(vocab_size, dim)).astype("float32")
    found = 0
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        vec = vectors.get(word)
        if vec is not None:
            matrix[idx] = vec
            found += 1
    coverage = found / min(len(word_index), vocab_size)
    return matrix, coverage


try:
    embedding_matrix, coverage_ratio = build_embedding_matrix(
        sequence_tokenizer.word_index, vocab_size, CFG.EMBEDDING_DIM, CFG.EMBEDDING_SOURCE
    )
    print(f"Embedding coverage ({CFG.EMBEDDING_SOURCE}): {coverage_ratio:.1%} of vocabulary")
except FileNotFoundError as err:
    print(f"[warning] {err}\nFalling back to randomly-initialized, trainable embeddings.")
    embedding_matrix = np.random.normal(
        scale=0.1, size=(vocab_size, CFG.EMBEDDING_DIM)
    ).astype("float32")
    coverage_ratio = 0.0


## 9 · Recurrent Baselines: LSTM and GRU

Both baselines share the same embedding layer factory and the same training/evaluation
harness — only the recurrent cell type differs — so any accuracy gap we see is attributable
to the cell, not to inconsistent setup.

In [ ]:
def make_embedding_layer(trainable: bool = True) -> layers.Embedding:
    return layers.Embedding(
        input_dim=vocab_size,
        output_dim=CFG.EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=CFG.MAX_SEQ_LEN,
        trainable=trainable,
        mask_zero=True,
        name="pretrained_embedding",
    )


In [ ]:
def compile_classifier(model: tf.keras.Model) -> tf.keras.Model:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CFG.LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def default_callbacks(checkpoint_name: str) -> list:
    return [
        callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        callbacks.ModelCheckpoint(
            f"{CFG.ARTIFACT_DIR}/{checkpoint_name}.keras",
            monitor="val_loss", save_best_only=True,
        ),
    ]


In [ ]:
def build_lstm_classifier() -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=(CFG.MAX_SEQ_LEN,)),
        make_embedding_layer(),
        layers.LSTM(CFG.RNN_UNITS, dropout=CFG.DROPOUT_RATE, recurrent_dropout=0.0),
        layers.Dense(64, activation="relu"),
        layers.Dropout(CFG.DROPOUT_RATE),
        layers.Dense(len(TARGET_EMOTIONS), activation="softmax", dtype="float32"),
    ], name="lstm_classifier")
    return compile_classifier(model)


lstm_model = build_lstm_classifier()
lstm_model.summary()


In [ ]:
def train_sequence_model(model: tf.keras.Model, checkpoint_name: str):
    started = time.time()
    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        batch_size=CFG.BATCH_SIZE,
        epochs=CFG.EPOCHS,
        callbacks=default_callbacks(checkpoint_name),
        verbose=2,
    )
    elapsed = time.time() - started
    return history, elapsed


lstm_history, lstm_train_seconds = train_sequence_model(lstm_model, "lstm_best")


In [ ]:
def plot_training_curves(history, title: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title(f"{title} — loss")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title(f"{title} — accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"{CFG.FIGURE_DIR}/{title.lower().replace(' ', '_')}_curves.png")
    plt.show()


plot_training_curves(lstm_history, "LSTM")


### 9.1 GRU baseline

Identical harness, swapping only the recurrent cell.

In [ ]:
def build_gru_classifier() -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=(CFG.MAX_SEQ_LEN,)),
        make_embedding_layer(),
        layers.GRU(CFG.RNN_UNITS, dropout=CFG.DROPOUT_RATE, recurrent_dropout=0.0),
        layers.Dense(64, activation="relu"),
        layers.Dropout(CFG.DROPOUT_RATE),
        layers.Dense(len(TARGET_EMOTIONS), activation="softmax", dtype="float32"),
    ], name="gru_classifier")
    return compile_classifier(model)


gru_model = build_gru_classifier()
gru_model.summary()


In [ ]:
gru_history, gru_train_seconds = train_sequence_model(gru_model, "gru_best")
plot_training_curves(gru_history, "GRU")


## 10 · A Hand-Rolled Additive Attention Layer

Rather than importing a black-box attention block, we implement **Bahdanau-style additive
attention** from first principles so we can (a) understand exactly what it computes and
(b) pull the attention weights back out later for visualization.

**The math.** Given per-timestep hidden states $h_1, \dots, h_T$ from a BiLSTM, we compute an
unnormalized score for each timestep:

$$e_t = v^\top \tanh(W h_t + b)$$

then normalize across the sequence with softmax to get attention weights:

$$\alpha_t = \frac{\exp(e_t)}{\sum_{k=1}^{T} \exp(e_k)}$$

and finally take the weighted sum of hidden states as the context vector fed into the
classifier head:

$$c = \sum_{t=1}^{T} \alpha_t \, h_t$$

Intuitively: the layer learns which timesteps (words) matter most for the final prediction,
and $\alpha_t$ is directly interpretable as "how much attention did token $t$ get".

In [ ]:
class AdditiveAttention(layers.Layer):
    '''Bahdanau-style additive attention with weights exposed for inspection.'''

    def __init__(self, units: int, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.score_dense = layers.Dense(units, activation="tanh", name="attn_score_proj")
        self.context_vector = layers.Dense(1, use_bias=False, name="attn_context_vec")

    def call(self, hidden_states, mask=None):
        # hidden_states: (batch, timesteps, hidden_dim)
        scores = self.context_vector(self.score_dense(hidden_states))  # (batch, T, 1)
        scores = tf.squeeze(scores, axis=-1)  # (batch, T)

        if mask is not None:
            scores = tf.where(mask, scores, tf.fill(tf.shape(scores), -1e9))

        weights = tf.nn.softmax(scores, axis=-1)  # alpha_t, (batch, T)
        weighted_states = hidden_states * tf.expand_dims(weights, axis=-1)
        context = tf.reduce_sum(weighted_states, axis=1)  # (batch, hidden_dim)
        return context, weights

    def compute_mask(self, inputs, mask=None):
        return None  # context vector has no time dimension left to mask


## 11 · BiLSTM + Attention

We build this one with the **Functional API** rather than `Sequential`, because we need two
outputs (the softmax prediction *and* the raw attention weights) for the visualization step
later.

In [ ]:
def build_bilstm_attention_classifier() -> tuple[tf.keras.Model, tf.keras.Model]:
    sequence_input = layers.Input(shape=(CFG.MAX_SEQ_LEN,), name="token_ids")
    embedded = make_embedding_layer()(sequence_input)

    bilstm_out = layers.Bidirectional(
        layers.LSTM(CFG.RNN_UNITS, dropout=CFG.DROPOUT_RATE, return_sequences=True),
        name="bidirectional_encoder",
    )(embedded)

    attention_layer = AdditiveAttention(CFG.ATTENTION_UNITS, name="additive_attention")
    context_vector, attention_weights = attention_layer(bilstm_out)

    dense = layers.Dense(64, activation="relu")(context_vector)
    dense = layers.Dropout(CFG.DROPOUT_RATE)(dense)
    output = layers.Dense(len(TARGET_EMOTIONS), activation="softmax", dtype="float32",
                           name="emotion_probs")(dense)

    training_model = models.Model(sequence_input, output, name="bilstm_attention_classifier")
    # a second head sharing all weights, used only for pulling out attention maps at inference
    inspection_model = models.Model(sequence_input, [output, attention_weights],
                                     name="bilstm_attention_inspector")
    return compile_classifier(training_model), inspection_model


bilstm_attn_model, bilstm_attn_inspector = build_bilstm_attention_classifier()
bilstm_attn_model.summary()


In [ ]:
bilstm_attn_history, bilstm_attn_train_seconds = train_sequence_model(
    bilstm_attn_model, "bilstm_attention_best"
)
plot_training_curves(bilstm_attn_history, "BiLSTM + Attention")


## 12 · Token-Level Attention Inspection

For any input sentence, we can now show exactly which tokens the BiLSTM+Attention model
leaned on to make its prediction.

In [ ]:
def inspect_attention(sentence: str, top_k: int = 5):
    cleaned = normalize_text(sentence)
    padded = texts_to_padded(pd.Series([cleaned]))
    probs, attn_weights = bilstm_attn_inspector.predict(padded, verbose=0)

    tokens = cleaned.split()[:CFG.MAX_SEQ_LEN]
    weights = attn_weights[0][:len(tokens)]
    weights = weights / (weights.sum() + 1e-9)  # renormalize over the *actual* tokens only

    predicted_idx = int(np.argmax(probs[0]))
    result = {
        "sentence": sentence,
        "predicted_emotion": id_to_emotion[predicted_idx],
        "confidence": float(probs[0][predicted_idx]),
        "tokens": tokens,
        "attention": weights.tolist(),
    }
    return result


def render_attention_heatmap(result: dict) -> None:
    tokens, weights = result["tokens"], np.array(result["attention"]).reshape(1, -1)
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.6), 2))
    sns.heatmap(weights, annot=False, cmap="YlOrRd", cbar=True,
                xticklabels=tokens, yticklabels=["attention"], ax=ax)
    ax.set_title(
        f'"{result["sentence"]}" → {result["predicted_emotion"]} '
        f'({result["confidence"]:.1%} confidence)'
    )
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


demo_result = inspect_attention("I can't believe you did that, I'm so hurt and angry")
render_attention_heatmap(demo_result)


## 13 · Fine-Tuning DistilBERT

Switching to the Hugging Face `Trainer` API for the transformer branch — it already handles
mixed precision, gradient accumulation, and checkpointing conventions that would otherwise be
reinvented by hand.

In [ ]:
from datasets import Dataset as HFDataset

def to_hf_dataset(df: pd.DataFrame) -> HFDataset:
    return HFDataset.from_pandas(
        df[["clean_text", "label_id"]].rename(columns={"clean_text": "text", "label_id": "label"}),
        preserve_index=False,
    )


hf_train = to_hf_dataset(clean_train)
hf_val   = to_hf_dataset(clean_val)
hf_test  = to_hf_dataset(clean_test)
hf_train


In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(CFG.TRANSFORMER_CHECKPOINT)


def tokenize_batch(batch):
    return bert_tokenizer(
        batch["text"], truncation=True, max_length=CFG.TRANSFORMER_MAX_LEN, padding=False
    )


hf_train_tok = hf_train.map(tokenize_batch, batched=True, remove_columns=["text"])
hf_val_tok   = hf_val.map(tokenize_batch, batched=True, remove_columns=["text"])
hf_test_tok  = hf_test.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)


In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.TRANSFORMER_CHECKPOINT,
    num_labels=len(TARGET_EMOTIONS),
    id2label=id_to_emotion,
    label2id=emotion_to_id,
)


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "macro_f1": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"],
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=f"{CFG.ARTIFACT_DIR}/distilbert_run",
    learning_rate=CFG.TRANSFORMER_LR,
    per_device_train_batch_size=CFG.TRANSFORMER_BATCH_SIZE,
    per_device_eval_batch_size=CFG.TRANSFORMER_BATCH_SIZE,
    num_train_epochs=CFG.TRANSFORMER_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=CFG.SEED,
)

bert_trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_val_tok,
    processing_class=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_transformer_metrics,
)


In [ ]:
bert_train_started = time.time()
bert_trainer.train()
bert_train_seconds = time.time() - bert_train_started


In [ ]:
bert_trainer.save_model(f"{CFG.ARTIFACT_DIR}/distilbert_best")
bert_eval_metrics = bert_trainer.evaluate(hf_test_tok)
bert_eval_metrics


## 14 · Cross-Model Benchmark

A single harness computes the same metric set for every model — recurrent models via a Keras
predict pass, DistilBERT via the `Trainer` — so the comparison table below is apples-to-apples.

In [ ]:
def count_keras_params(model: tf.keras.Model) -> int:
    return int(np.sum([tf.size(w).numpy() for w in model.trainable_weights]))


def estimate_keras_memory_mb(model: tf.keras.Model) -> float:
    total_params = model.count_params()
    return total_params * 4 / (1024 ** 2)  # float32 assumption


def benchmark_keras_model(model: tf.keras.Model, name: str, train_seconds: float) -> dict:
    start = time.time()
    probs = model.predict(X_test_seq, batch_size=256, verbose=0)
    inference_seconds = time.time() - start

    preds = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average="macro", zero_division=0
    )
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "n_parameters": count_keras_params(model),
        "approx_memory_mb": estimate_keras_memory_mb(model),
        "_predictions": preds,
    }


In [ ]:
def benchmark_transformer(trainer: Trainer, dataset, name: str, train_seconds: float) -> dict:
    start = time.time()
    output = trainer.predict(dataset)
    inference_seconds = time.time() - start

    preds = np.argmax(output.predictions, axis=1)
    labels = np.array(dataset["label"])
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    n_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    approx_mb = n_params * 4 / (1024 ** 2)
    return {
        "model": name,
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "n_parameters": n_params,
        "approx_memory_mb": approx_mb,
        "_predictions": preds,
    }


In [ ]:
benchmark_rows = [
    benchmark_keras_model(lstm_model, "LSTM", lstm_train_seconds),
    benchmark_keras_model(gru_model, "GRU", gru_train_seconds),
    benchmark_keras_model(bilstm_attn_model, "BiLSTM + Attention", bilstm_attn_train_seconds),
    benchmark_transformer(bert_trainer, hf_test_tok, "DistilBERT (fine-tuned)", bert_train_seconds),
]

benchmark_df = pd.DataFrame(benchmark_rows).drop(columns="_predictions")
benchmark_df = benchmark_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)
benchmark_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
benchmark_df.set_index("model")[["accuracy", "macro_f1"]].plot(kind="bar", ax=ax)
ax.set_title("Accuracy vs. Macro-F1 across all four models")
ax.set_ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{CFG.FIGURE_DIR}/model_comparison.png")
plt.show()


## 15 · Confusion Matrices

Aggregate metrics hide *which* emotions get confused for which — e.g. we'd expect `fear` and
`surprise` to bleed into each other more than `joy` and `disgust`.

In [ ]:
def plot_confusion(y_true, y_pred, title: str) -> None:
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(TARGET_EMOTIONS))))
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=TARGET_EMOTIONS, yticklabels=TARGET_EMOTIONS, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {title}")
    plt.tight_layout()
    plt.savefig(f"{CFG.FIGURE_DIR}/confusion_{title.lower().replace(' ', '_').replace('+','')}.png")
    plt.show()


for row in benchmark_rows:
    plot_confusion(y_test, row["_predictions"], row["model"])


## 16 · Per-Class Classification Reports

In [ ]:
for row in benchmark_rows:
    print(f"\n=== {row['model']} ===")
    print(classification_report(y_test, row["_predictions"], target_names=TARGET_EMOTIONS,
                                 zero_division=0))


## 17 · Error Forensics

Rather than stopping at aggregate scores, we pull out the most-confident *wrong* predictions
from our best model — these are the cases most worth reading by hand.

In [ ]:
def build_error_frame(df_source: pd.DataFrame, probs: np.ndarray, model_name: str) -> pd.DataFrame:
    preds = np.argmax(probs, axis=1)
    confidence = np.max(probs, axis=1)
    errors = df_source.copy()
    errors["predicted"] = [id_to_emotion[p] for p in preds]
    errors["confidence"] = confidence
    errors["is_correct"] = errors["emotion"] == errors["predicted"]
    errors["model"] = model_name
    return errors[~errors["is_correct"]].sort_values("confidence", ascending=False)


best_model_name = benchmark_df.iloc[0]["model"]
print(f"Running error analysis on the top-ranked model: {best_model_name}")

if best_model_name == "DistilBERT (fine-tuned)":
    best_probs = torch.softmax(torch.tensor(bert_trainer.predict(hf_test_tok).predictions), dim=1).numpy()
else:
    model_lookup = {"LSTM": lstm_model, "GRU": gru_model, "BiLSTM + Attention": bilstm_attn_model}
    best_probs = model_lookup[best_model_name].predict(X_test_seq, batch_size=256, verbose=0)

error_frame = build_error_frame(clean_test, best_probs, best_model_name)
error_frame[["text", "emotion", "predicted", "confidence"]].head(10)


**Reading the errors:** high-confidence misclassifications are usually one of three
patterns — sarcasm the model reads literally, genuinely mixed-emotion text we forced into a
single label during collapse, or short comments with too little context to disambiguate
(e.g. "oh great" reading as `joy` on tone alone).

## 18 · Interactive Inference Playground

One function, all four models, one comparison table.

In [ ]:
def predict_all_models(sentence: str) -> pd.DataFrame:
    cleaned = normalize_text(sentence)
    padded = texts_to_padded(pd.Series([cleaned]))

    rows = []
    for name, keras_model in [("LSTM", lstm_model), ("GRU", gru_model),
                               ("BiLSTM + Attention", bilstm_attn_model)]:
        probs = keras_model.predict(padded, verbose=0)[0]
        rows.append({"model": name, "prediction": id_to_emotion[int(np.argmax(probs))],
                      "confidence": float(np.max(probs))})

    bert_inputs = bert_tokenizer(cleaned, return_tensors="pt", truncation=True,
                                  max_length=CFG.TRANSFORMER_MAX_LEN)
    with torch.no_grad():
        logits = bert_model(**bert_inputs).logits
    bert_probs = torch.softmax(logits, dim=1).numpy()[0]
    rows.append({"model": "DistilBERT (fine-tuned)",
                 "prediction": id_to_emotion[int(np.argmax(bert_probs))],
                 "confidence": float(np.max(bert_probs))})

    return pd.DataFrame(rows)


predict_all_models("I honestly did not expect that at all, what a twist!")


In [ ]:
# Try your own sentence here
predict_all_models("Type your own sentence to compare all four models")


## 19 · Final Conclusion

**Best performer:** see the top row of `benchmark_df` above (typically DistilBERT on
macro-F1, with BiLSTM+Attention closest among the recurrent family).

**Trade-offs observed:**

| Model | Strengths | Weaknesses |
|---|---|---|
| LSTM | Fast to train, tiny footprint | Weakest on rare classes (fear, disgust) |
| GRU | Fewer parameters than LSTM, similar accuracy | Same context-length limitations as LSTM |
| BiLSTM + Attention | Interpretable via attention maps, bidirectional context | Still struggles with sarcasm / mixed emotion |
| DistilBERT | Best macro-F1, strongest on rare classes | Largest memory footprint, slowest inference |

**Future improvements:**
- Re-introduce the dropped multi-label examples as a proper multi-label objective
- Try class-weighted loss or focal loss to further help `fear` / `disgust`
- Distill the fine-tuned DistilBERT back into the BiLSTM+Attention model for a fast,
  interpretable, near-Transformer-quality classifier
- Expand attention inspection into a full saliency comparison against DistilBERT's own
  attention heads

---

### 📐 Pipeline recap

```
Raw GoEmotions (27 labels, multi-label)
        │
        ▼
Six-class collapse (TAXONOMY_MAP) ──► drop unmapped / multi-label rows
        │
        ▼
normalize_text() ──► lowercase, expand contractions, strip URL/HTML/punct/digits
        │
        ├──► Tokenizer + padding ──► LSTM / GRU / BiLSTM+Attention
        │
        └──► DistilBERT tokenizer ──► HF Trainer fine-tune
                        │
                        ▼
        benchmark_df (accuracy, macro-P/R/F1, timing, params, memory)
                        │
                        ▼
        confusion matrices + classification reports + error forensics
                        │
                        ▼
                predict_all_models() playground
```
